# M12c · Valeurs propres et matrices symétriques — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m12c_valeurs_propres/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m12c_valeurs_propres/03_notebook.ipynb`, créée par `python tools/start_chapter.py M12c`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M12c.1 à M12c.13, M12c.20, M12c.21 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M12c.26 | Le déterminant par élimination · `det` | 🔨 | ★ | 20 |
|  | M12c.27 | La méthode de la puissance et le quotient de Rayleigh · `power_iteration, rayleigh_quotient` | 🔨 | ★★ | 25 |
|  | M12c.28 | La décomposition de Cholesky et le test « définie positive » · `cholesky, is_positive_definite` | 🔨 | ★★ | 25 |
|  | M12c.29 | La racine carrée d'une matrice symétrique positive et la matrice de blanchiment · `sqrtm_psd, whitening_matrix` | 🔨 | ★★ | 25 |
|  | M12c.30 | `np.linalg.eigh`, `det`, `cholesky` et `scipy.linalg.sqrtm` contre tes fonctions | 📦 | ★ | 15 |
| B | M12c.31 | Prédire : la méthode de la puissance quand les deux plus grandes valeurs propres sont proches | 🔮 | ★ | 10 |
|  | M12c.32 | La vitesse de la méthode de la puissance et le rapport \|λ₂ / λ₁\| | 🔬 | ★★ | 20 |
|  | M12c.33 | Les lignes de niveau de xᵀAx : définie positive, semi-définie positive, indéfinie | 🎨 | ★ | 20 |
| C | M12c.34 | Chasse à l'erreur : det(A + B) = det A + det B, un vecteur propre nul, Cholesky d'une matrice non symétrique | 🐛 | ★ | 15 |
|  | M12c.35 | Vérifier avec SymPy : polynôme caractéristique, valeurs propres exactes, diagonalisation | 🛠️ | ★ | 15 |
|  | M12c.36 | Défi : les trois plus grandes valeurs propres d'une matrice symétrique semi-définie positive 300 × 300, par la puissance avec déflation, à 1e−6 près | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : déterminants, valeurs et vecteurs propres, diagonalisation et puissances, matrices défectives et rotations, théorème spectral, quotient de Rayleigh, méthode de la puissance, matrices définies positives et hessienne, Cholesky, formes quadratiques, covariance transformée et blanchiment.
- Écrire les sept fonctions de `mymath/eigen.py` et les tester, puis comparer avec NumPy et SciPy.
- Prédire et mesurer la vitesse de la méthode de la puissance, dessiner des formes quadratiques, chasser des erreurs, vérifier avec SymPy, trouver les plus grandes valeurs propres d'une grande matrice.

**Rappel express.** $\det$ : volume orienté, $\det(\mathbf{A}\mathbf{B}) = \det\mathbf{A}\det\mathbf{B}$, $\det\mathbf{A} \ne 0 \iff$ inversible ; $\operatorname{tr}(\mathbf{A}\mathbf{B}) = \operatorname{tr}(\mathbf{B}\mathbf{A})$ ; $\mathbf{A}\mathbf{v} = \lambda\mathbf{v}$, $\det(\mathbf{A} - \lambda\mathbf{I}) = 0$, $\sum\lambda_i = \operatorname{tr}\mathbf{A}$, $\prod\lambda_i = \det\mathbf{A}$ ; $\mathbf{A} = \mathbf{P}\mathbf{D}\mathbf{P}^{-1}$, $\mathbf{A}^k = \mathbf{P}\mathbf{D}^k\mathbf{P}^{-1}$ ; puissance : $\mathbf{x} \leftarrow \mathbf{A}\mathbf{x}/\lVert \mathbf{A}\mathbf{x} \rVert$, vitesse $|\lambda_2/\lambda_1|$ ; symétrique : $\mathbf{A} = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^\top$, $\lambda_{\min} \le R(\mathbf{x}) \le \lambda_{\max}$ ; définie positive $\iff$ $\lambda_i > 0$ $\iff$ $\mathbf{A} = \mathbf{L}\mathbf{L}^\top$ ; $\operatorname{Cov}(\mathbf{A}\mathbf{x}) = \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^\top$ ; blanchir : $\mathbf{W}\boldsymbol{\Sigma}\mathbf{W}^\top = \mathbf{I}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M12c.1 à M12c.13, 🔗 M12c.20 et M12c.21)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0, -2, 5]` pour un vecteur (les valeurs propres par ordre décroissant), une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex M12c.1 — Déterminants 2 × 2 et 3 × 3 : aire, volume et signe**

In [ ]:
# Ex M12c.1: the values you found on paper
answer_M12c_1a = ...  # a) a number
answer_M12c_1b = ...  # b) a number
answer_M12c_1c = ...  # c) a number (2 decimals)
answer_M12c_1d = ...  # d) a number
answer_M12c_1e = ...  # e) True or False
answer_M12c_1f = ...  # f) a number
answer_M12c_1g = ...  # g) a number
answer_M12c_1h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M12c_1a, answer_M12c_1b, answer_M12c_1c, answer_M12c_1d, answer_M12c_1e, answer_M12c_1f, answer_M12c_1g, answer_M12c_1h]):
    wbm.check(f"M12c.1{letter}", answer)

**Ex M12c.2 — Le déterminant d'une matrice 4 × 4 par élimination**

In [ ]:
# Ex M12c.2: the values you found on paper
answer_M12c_2a = ...  # a) a matrix
answer_M12c_2b = ...  # b) a list of 4 numbers (1 decimal)
answer_M12c_2c = ...  # c) a number
answer_M12c_2d = ...  # d) a number (4 decimals)
answer_M12c_2e = ...  # e) a number
answer_M12c_2f = ...  # f) a number
answer_M12c_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M12c_2a, answer_M12c_2b, answer_M12c_2c, answer_M12c_2d, answer_M12c_2e, answer_M12c_2f, answer_M12c_2g]):
    wbm.check(f"M12c.2{letter}", answer)

**Ex M12c.3 — Les valeurs et vecteurs propres d'une matrice 2 × 2**

In [ ]:
# Ex M12c.3: the values you found on paper
answer_M12c_3a = ...  # a) a number
answer_M12c_3b = ...  # b) a number
answer_M12c_3c = ...  # c) a list of 2 numbers
answer_M12c_3d = ...  # d) a list of 2 numbers
answer_M12c_3e = ...  # e) a list of 2 numbers
answer_M12c_3f = ...  # f) a list of 2 numbers
answer_M12c_3g = ...  # g) a list of 2 numbers
answer_M12c_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_3a, answer_M12c_3b, answer_M12c_3c, answer_M12c_3d, answer_M12c_3e, answer_M12c_3f, answer_M12c_3g, answer_M12c_3h]):
    wbm.check(f"M12c.3{letter}", answer)

**Ex M12c.4 — Une matrice 3 × 3 triangulaire, puis une avec une valeur propre double**

In [ ]:
# Ex M12c.4: the values you found on paper
answer_M12c_4a = ...  # a) a list of 3 numbers
answer_M12c_4b = ...  # b) a list of 3 numbers
answer_M12c_4c = ...  # c) a list of 3 numbers
answer_M12c_4d = ...  # d) a number
answer_M12c_4e = ...  # e) a list of 3 numbers
answer_M12c_4f = ...  # f) an integer
answer_M12c_4g = ...  # g) a list of 3 numbers
answer_M12c_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_4a, answer_M12c_4b, answer_M12c_4c, answer_M12c_4d, answer_M12c_4e, answer_M12c_4f, answer_M12c_4g, answer_M12c_4h]):
    wbm.check(f"M12c.4{letter}", answer)

**Ex M12c.5 — Diagonaliser, puis calculer une puissance dixième**

In [ ]:
# Ex M12c.5: the values you found on paper
answer_M12c_5a = ...  # a) a list of 2 numbers
answer_M12c_5b = ...  # b) a list of 2 numbers
answer_M12c_5c = ...  # c) a list of 2 numbers
answer_M12c_5d = ...  # d) a matrix
answer_M12c_5e = ...  # e) a matrix
answer_M12c_5f = ...  # f) a number
answer_M12c_5g = ...  # g) a number
answer_M12c_5h = ...  # h) a list of 2 numbers (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M12c_5a, answer_M12c_5b, answer_M12c_5c, answer_M12c_5d, answer_M12c_5e, answer_M12c_5f, answer_M12c_5g, answer_M12c_5h]):
    wbm.check(f"M12c.5{letter}", answer)

**Ex M12c.6 — Une matrice non diagonalisable, et une rotation sans valeur propre réelle**

In [ ]:
# Ex M12c.6: the values you found on paper
answer_M12c_6a = ...  # a) a list of 2 numbers
answer_M12c_6b = ...  # b) an integer
answer_M12c_6c = ...  # c) a list of 2 numbers
answer_M12c_6d = ...  # d) True or False
answer_M12c_6e = ...  # e) a number
answer_M12c_6f = ...  # f) an integer
answer_M12c_6g = ...  # g) a number
answer_M12c_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_6a, answer_M12c_6b, answer_M12c_6c, answer_M12c_6d, answer_M12c_6e, answer_M12c_6f, answer_M12c_6g, answer_M12c_6h]):
    wbm.check(f"M12c.6{letter}", answer)

**Ex M12c.7 — Le théorème spectral sur une matrice symétrique 2 × 2 : Q, Λ et QΛQᵀ**

In [ ]:
# Ex M12c.7: the values you found on paper
answer_M12c_7a = ...  # a) a list of 2 numbers
answer_M12c_7b = ...  # b) a list of 2 numbers (4 decimals)
answer_M12c_7c = ...  # c) a list of 2 numbers (4 decimals)
answer_M12c_7d = ...  # d) a number
answer_M12c_7e = ...  # e) a matrix (4 decimals)
answer_M12c_7f = ...  # f) a number
answer_M12c_7g = ...  # g) a list of 2 numbers (4 decimals)
answer_M12c_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_7a, answer_M12c_7b, answer_M12c_7c, answer_M12c_7d, answer_M12c_7e, answer_M12c_7f, answer_M12c_7g, answer_M12c_7h]):
    wbm.check(f"M12c.7{letter}", answer)

**Ex M12c.8 — Le quotient de Rayleigh : le maximum et le minimum de xᵀAx sur le cercle unité**

In [ ]:
# Ex M12c.8: the values you found on paper
answer_M12c_8a = ...  # a) a number
answer_M12c_8b = ...  # b) a number
answer_M12c_8c = ...  # c) a number (1 decimal)
answer_M12c_8d = ...  # d) a list of 2 numbers
answer_M12c_8e = ...  # e) a number
answer_M12c_8f = ...  # f) a list of 2 numbers (4 decimals)
answer_M12c_8g = ...  # g) a number
answer_M12c_8h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12c_8a, answer_M12c_8b, answer_M12c_8c, answer_M12c_8d, answer_M12c_8e, answer_M12c_8f, answer_M12c_8g, answer_M12c_8h]):
    wbm.check(f"M12c.8{letter}", answer)

**Ex M12c.9 — Trois itérations de la méthode de la puissance à la main**

In [ ]:
# Ex M12c.9: the values you found on paper
answer_M12c_9a = ...  # a) a list of 2 numbers
answer_M12c_9b = ...  # b) a list of 2 numbers (4 decimals)
answer_M12c_9c = ...  # c) a list of 2 numbers (4 decimals)
answer_M12c_9d = ...  # d) a list of 2 numbers (4 decimals)
answer_M12c_9e = ...  # e) a number (4 decimals)
answer_M12c_9f = ...  # f) a list of 2 numbers
answer_M12c_9g = ...  # g) a list of 2 numbers (4 decimals)
answer_M12c_9h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12c_9a, answer_M12c_9b, answer_M12c_9c, answer_M12c_9d, answer_M12c_9e, answer_M12c_9f, answer_M12c_9g, answer_M12c_9h]):
    wbm.check(f"M12c.9{letter}", answer)

**Ex M12c.10 — Définie positive ? Par les valeurs propres, les mineurs et la forme quadratique ; une hessienne qui prouve la convexité**

In [ ]:
# Ex M12c.10: the values you found on paper
answer_M12c_10a = ...  # a) a list of 3 numbers
answer_M12c_10b = ...  # b) True or False
answer_M12c_10c = ...  # c) a list of 3 numbers
answer_M12c_10d = ...  # d) a number
answer_M12c_10e = ...  # e) a matrix
answer_M12c_10f = ...  # f) a number (4 decimals)
answer_M12c_10g = ...  # g) True or False
answer_M12c_10h = ...  # h) a list of 2 numbers

for letter, answer in zip("abcdefgh", [answer_M12c_10a, answer_M12c_10b, answer_M12c_10c, answer_M12c_10d, answer_M12c_10e, answer_M12c_10f, answer_M12c_10g, answer_M12c_10h]):
    wbm.check(f"M12c.10{letter}", answer)

**Ex M12c.11 — La décomposition de Cholesky d'une matrice 3 × 3**

In [ ]:
# Ex M12c.11: the values you found on paper
answer_M12c_11a = ...  # a) a number
answer_M12c_11b = ...  # b) a list of 2 numbers
answer_M12c_11c = ...  # c) a number
answer_M12c_11d = ...  # d) a number
answer_M12c_11e = ...  # e) a number
answer_M12c_11f = ...  # f) a matrix
answer_M12c_11g = ...  # g) a number
answer_M12c_11h = ...  # h) a list of 3 numbers

for letter, answer in zip("abcdefgh", [answer_M12c_11a, answer_M12c_11b, answer_M12c_11c, answer_M12c_11d, answer_M12c_11e, answer_M12c_11f, answer_M12c_11g, answer_M12c_11h]):
    wbm.check(f"M12c.11{letter}", answer)

**Ex M12c.12 — Une forme quadratique réduite à une somme de carrés, et ses lignes de niveau**

In [ ]:
# Ex M12c.12: the values you found on paper
answer_M12c_12a = ...  # a) a matrix
answer_M12c_12b = ...  # b) a list of 2 numbers
answer_M12c_12c = ...  # c) a list of 2 numbers (4 decimals)
answer_M12c_12d = ...  # d) a number
answer_M12c_12e = ...  # e) a list of 2 numbers (4 decimals)
answer_M12c_12f = ...  # f) a number (2 decimals)
answer_M12c_12g = ...  # g) a number (2 decimals)
answer_M12c_12h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_12a, answer_M12c_12b, answer_M12c_12c, answer_M12c_12d, answer_M12c_12e, answer_M12c_12f, answer_M12c_12g, answer_M12c_12h]):
    wbm.check(f"M12c.12{letter}", answer)

**Ex M12c.13 — La covariance d'un vecteur transformé, AΣAᵀ, et le blanchiment d'une covariance 2 × 2**

In [ ]:
# Ex M12c.13: the values you found on paper
answer_M12c_13a = ...  # a) a number
answer_M12c_13b = ...  # b) a matrix
answer_M12c_13c = ...  # c) a matrix
answer_M12c_13d = ...  # d) a matrix (4 decimals)
answer_M12c_13e = ...  # e) a list of 2 numbers
answer_M12c_13f = ...  # f) a matrix (4 decimals)
answer_M12c_13g = ...  # g) a matrix (4 decimals)
answer_M12c_13h = ...  # h) a list of 2 numbers (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12c_13a, answer_M12c_13b, answer_M12c_13c, answer_M12c_13d, answer_M12c_13e, answer_M12c_13f, answer_M12c_13g, answer_M12c_13h]):
    wbm.check(f"M12c.13{letter}", answer)

**Ex M12c.20 — Pont : la variance d'une somme pondérée de trois features, wᵀΣw, et pourquoi Σ doit être semi-définie positive**

In [ ]:
# Ex M12c.20: the values you found on paper
answer_M12c_20a = ...  # a) a number
answer_M12c_20b = ...  # b) a number
answer_M12c_20c = ...  # c) a number
answer_M12c_20d = ...  # d) a number
answer_M12c_20e = ...  # e) a number
answer_M12c_20f = ...  # f) a number
answer_M12c_20g = ...  # g) True or False
answer_M12c_20h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M12c_20a, answer_M12c_20b, answer_M12c_20c, answer_M12c_20d, answer_M12c_20e, answer_M12c_20f, answer_M12c_20g, answer_M12c_20h]):
    wbm.check(f"M12c.20{letter}", answer)

**Ex M12c.21 — Pont : après un blanchiment, la covariance est l'identité**

In [ ]:
# Ex M12c.21: the values you found on paper
answer_M12c_21a = ...  # a) a matrix
answer_M12c_21b = ...  # b) a matrix (4 decimals)
answer_M12c_21c = ...  # c) a matrix
answer_M12c_21d = ...  # d) a list of 2 numbers (1 decimal)
answer_M12c_21e = ...  # e) a number (1 decimal)
answer_M12c_21f = ...  # f) a matrix
answer_M12c_21g = ...  # g) True or False
answer_M12c_21h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M12c_21a, answer_M12c_21b, answer_M12c_21c, answer_M12c_21d, answer_M12c_21e, answer_M12c_21f, answer_M12c_21g, answer_M12c_21h]):
    wbm.check(f"M12c.21{letter}", answer)

## Partie A · Ta librairie : déterminant, puissance, Cholesky, racine carrée et blanchiment ; NumPy

Fiche §1, §5 à §8. Les sept fonctions de `mymath/eigen.py`, puis les mêmes calculs avec NumPy et SciPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import StopCheck, checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def random_spd(n, seed, shift=0.5):
    """A random symmetric positive definite (n, n) matrix, drawn with default_rng(seed)."""
    X = np.random.default_rng(seed).normal(size=(n + 3, n))
    return X.T @ X / (n + 3) + shift * np.eye(n)

### Ex M12c.26 — Le déterminant par élimination · `det` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le déterminant par élimination avec pivot partiel, en suivant le signe des échanges.  
**Prérequis :** fiche §1 · **mymath :** `eigen.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/eigen.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis, `@`, `.T`, `np.linalg.norm` et `np.linalg.solve` compris ; `np.linalg.det`, `slogdet`, `eig`, `eigvals`, `cholesky`, `inv`, `scipy.linalg` et SymPy sont interdits (les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas), et `np.linalg.eigh` n'est permis que dans `sqrtm_psd` et `whitening_matrix`. Ne modifie jamais la matrice reçue : travaille sur une copie, `np.array(A, dtype=float)`. Les nombres se rendent en `float` de Python, les tests en `bool`.

Écris `det(A)` (fiche §1), en suivant la docstring : une copie en flottants, une `ValueError` si $\mathbf{A}$ n'est pas carrée, un signe qui vaut 1 au départ ; puis, pour chaque colonne $k$ : chercher, de la ligne $k$ à la dernière, l'élément de **plus grande valeur absolue** (`np.argmax(np.abs(...))`) ; s'il est nul, rendre `0.0` aussitôt ; sinon remonter sa ligne en position $k$ (et changer le signe si c'est un vrai échange), puis retirer aux lignes du dessous le bon multiple de la ligne $k$. Le déterminant est le signe fois le produit de la diagonale, en `float` de Python.

La vérification calcule :
- a) `det([[1, 3, 2], [-2, 0, 4], [3, 1, -1]])` (6 décimales) ;
- b) `det([[0, 2, 1], [3, -1, 2], [1, 1, 0]])` (6 décimales) ;
- c) vrai ou faux : pour une matrice $6 \times 6$ tirée au hasard (`np.random.default_rng(26)`, loi normale), `det` et `np.linalg.det` sont égaux à $10^{-10}$ près, relativement ;
- d) vrai ou faux : `d = det([[1, 2, 3], [2, 4, 6], [1, 0, 1]])` vérifie `type(d) is float and d == 0 and math.copysign(1.0, d) > 0` (un `float` de Python, égal à 0, et positif : pas `-0.0`) ;
- e) vrai ou faux : pour deux matrices $4 \times 4$ tirées au hasard (même générateur, après celle de c)), $\det(\mathbf{A}\mathbf{B}) = \det\mathbf{A}\,\det\mathbf{B}$ à $10^{-10}$ près, relativement ;
- f) le nom de l'erreur levée par `det([[1, 2, 3], [4, 5, 6]])` ;
- g) vrai ou faux : après `det(M)`, le tableau `M` (une copie en flottants de la matrice de b)) n'a pas changé.

Dans ta copie : retrouve a) et b) à la main. Pour b), quelle ligne est remontée en premier, et combien d'échanges y a-t-il ? Que rend `repr(det(...))` pour la matrice de d), et pourquoi ne doit-il afficher ni `-0.0` ni `np.float64(0.0)` ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")  # reload your saved file

In [ ]:
rng_26 = np.random.default_rng(26)
C_26 = rng_26.normal(size=(6, 6))
D_26, E_26 = rng_26.normal(size=(4, 4)), rng_26.normal(size=(4, 4))
with checked("M12c.26a"):
    wbm.check("M12c.26a", mymath.eigen.det([[1, 3, 2], [-2, 0, 4], [3, 1, -1]]), computed=True)
with checked("M12c.26b"):
    wbm.check("M12c.26b", mymath.eigen.det([[0, 2, 1], [3, -1, 2], [1, 1, 0]]), computed=True)
with checked("M12c.26c"):
    wbm.check("M12c.26c", bool(abs(mymath.eigen.det(C_26) - np.linalg.det(C_26)) <= 1e-10 * max(1.0, abs(np.linalg.det(C_26)))),
              computed=True)
with checked("M12c.26d"):
    d_26 = mymath.eigen.det([[1, 2, 3], [2, 4, 6], [1, 0, 1]])
    wbm.check("M12c.26d", bool(type(d_26) is float and d_26 == 0 and math.copysign(1.0, d_26) > 0), computed=True)
with checked("M12c.26e"):
    d_ab = mymath.eigen.det(D_26 @ E_26)
    wbm.check("M12c.26e", bool(abs(d_ab - mymath.eigen.det(D_26) * mymath.eigen.det(E_26)) <= 1e-10 * max(1.0, abs(d_ab))),
              computed=True)
with checked("M12c.26f"):
    wbm.check("M12c.26f", raised(mymath.eigen.det, [[1, 2, 3], [4, 5, 6]]), computed=True)
with checked("M12c.26g"):
    M_26 = np.array([[0, 2, 1], [3, -1, 2], [1, 1, 0]], dtype=float)
    mymath.eigen.det(M_26)
    wbm.check("M12c.26g", bool(np.array_equal(M_26, np.array([[0, 2, 1], [3, -1, 2], [1, 1, 0]], dtype=float))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12c_eigen.py", "test_det_ or test_eigen_uses_")

### Ex M12c.27 — La méthode de la puissance et le quotient de Rayleigh · `power_iteration, rayleigh_quotient` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la méthode de la puissance, avec sa normalisation, sa convention de signe et son critère d'arrêt, et le quotient de Rayleigh.  
**Prérequis :** fiche §5 et §6 ; Ex M12c.26 · **mymath :** `eigen.py` · **Parcours :** C, E

Écris, en suivant les docstrings :
- `rayleigh_quotient(A, x)` (fiche §6) : vérifie les formes et que $\mathbf{x} \ne \mathbf{0}$ (sinon `ValueError`), puis $\mathbf{x}^\top\mathbf{A}\mathbf{x}/\mathbf{x}^\top\mathbf{x}$ ;
- `power_iteration(A, x0=None, n_iter=1000, tol=1e-12)` (fiche §5) : $\mathbf{x}_0$ (des 1 par défaut) normalisé, avec la convention de signe (la coordonnée de plus grande valeur absolue positive) ; puis, au plus `n_iter` fois : $\mathbf{y} = \mathbf{A}\mathbf{x}$ (`ValueError` s'il est nul), $\mathbf{x}_{\text{new}} = \mathbf{y}/\lVert \mathbf{y} \rVert$ avec la convention de signe, et arrêt dès que $\lVert \mathbf{x}_{\text{new}} - \mathbf{x} \rVert \le$ `tol`. Rends `(valeur, vecteur, itérations)`, la valeur étant $\mathbf{x}^\top\mathbf{A}\mathbf{x}$ pour le dernier $\mathbf{x}$ (unitaire), et les itérations le nombre de produits $\mathbf{A}\mathbf{x}$ calculés.

La vérification calcule, avec $\mathbf{S}$ = `[[4, 1, 0], [1, 3, 1], [0, 1, 2]]` :
- a) la valeur propre rendue par `power_iteration(S)` (6 décimales) ;
- b) le vecteur propre rendu par `power_iteration(S)` (une liste, 6 décimales) ;
- c) `rayleigh_quotient(S, [1, 1, 1])` (6 décimales) ;
- d) vrai ou faux : pour une matrice symétrique définie positive $8 \times 8$ tirée au hasard (`random_spd(8, 27)`, des outils), la valeur rendue est la plus grande valeur propre de `np.linalg.eigvalsh` à $10^{-8}$ près, et $\lVert \mathbf{A}\mathbf{v} - \lambda\mathbf{v} \rVert \le 10^{-6}$ ;
- e) la valeur, le vecteur et le nombre d'itérations rendus par `power_iteration(np.diag([-5.0, 2.0, 1.0]), x0=[1, 1, 1])` (une liste de 5 nombres : la valeur, les trois coordonnées du vecteur, puis le nombre d'itérations, 6 décimales) ;
- f) le nom de l'erreur levée par `power_iteration([[1, 2], [3, 4]], x0=[0, 0])` ;
- g) le nom de l'erreur levée par `rayleigh_quotient([[1, 2], [3, 4]], [0, 0])`.

Dans ta copie : combien d'itérations demande a), et pourquoi est-ce cohérent avec le rapport des deux plus grandes valeurs propres de $\mathbf{S}$ (calcule-les avec `np.linalg.eigvalsh`) ? Pourquoi c) est-il proche de a), sans lui être égal ? Que ferait $\mathbf{x}$ dans e) sans la convention de signe ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")  # reload your saved file

In [ ]:
A_27 = random_spd(8, 27)
with checked("M12c.27a"):
    wbm.check("M12c.27a", mymath.eigen.power_iteration([[4, 1, 0], [1, 3, 1], [0, 1, 2]])[0], computed=True)
with checked("M12c.27b"):
    wbm.check("M12c.27b", np.asarray(mymath.eigen.power_iteration([[4, 1, 0], [1, 3, 1], [0, 1, 2]])[1]).tolist(), computed=True)
with checked("M12c.27c"):
    wbm.check("M12c.27c", mymath.eigen.rayleigh_quotient([[4, 1, 0], [1, 3, 1], [0, 1, 2]], [1, 1, 1]), computed=True)
with checked("M12c.27d"):
    value_27, vector_27, _ = mymath.eigen.power_iteration(A_27)
    vector_27 = np.asarray(vector_27, dtype=float)
    wbm.check("M12c.27d", bool(abs(value_27 - np.linalg.eigvalsh(A_27)[-1]) <= 1e-8
                               and np.linalg.norm(A_27 @ vector_27 - value_27 * vector_27) <= 1e-6), computed=True)
with checked("M12c.27e"):
    value_27e, vector_27e, iterations_27e = mymath.eigen.power_iteration(np.diag([-5.0, 2.0, 1.0]), x0=[1, 1, 1])
    wbm.check("M12c.27e", [value_27e, *np.asarray(vector_27e).tolist(), iterations_27e], computed=True)
with checked("M12c.27f"):
    wbm.check("M12c.27f", raised(mymath.eigen.power_iteration, [[1, 2], [3, 4]], x0=[0, 0]), computed=True)
with checked("M12c.27g"):
    wbm.check("M12c.27g", raised(mymath.eigen.rayleigh_quotient, [[1, 2], [3, 4]], [0, 0]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12c_eigen.py", "test_power_iteration_ or test_rayleigh_quotient_ or test_eigen_uses_")

### Ex M12c.28 — La décomposition de Cholesky et le test « définie positive » · `cholesky, is_positive_definite` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la décomposition de Cholesky colonne par colonne, avec ses contrôles, et s'en servir pour tester qu'une matrice est définie positive.  
**Prérequis :** fiche §7 ; Ex M12c.26 · **mymath :** `eigen.py` · **Parcours :** C, E

Écris, en suivant les docstrings :
- `cholesky(A)` (fiche §7) : une copie en flottants, une `ValueError` si $\mathbf{A}$ n'est pas carrée ou pas symétrique (au sens de la docstring du module : $|a_{ij} - a_{ji}| \le 10^{-12} \max(1, \max|\mathbf{A}|)$) ; puis, colonne par colonne, $l_{jj} = \sqrt{a_{jj} - \sum_{k<j} l_{jk}^2}$ (une `ValueError` si la quantité sous la racine n'est pas **strictement** positive) et $l_{ij} = (a_{ij} - \sum_{k<j} l_{ik}l_{jk})/l_{jj}$ sous la diagonale ; des zéros exacts au-dessus ;
- `is_positive_definite(A)` (fiche §7) : `ValueError` si $\mathbf{A}$ n'est pas carrée ; sinon essaie `cholesky` dans un `try`, et rends `True` ou `False` (des `bool` de Python).

La vérification calcule :
- a) `cholesky([[25, -10, 15], [-10, 5, -4], [15, -4, 17]])` (une matrice, 6 décimales) ;
- b) vrai ou faux : pour `random_spd(6, 28)`, $\mathbf{L}\mathbf{L}^\top = \mathbf{A}$ à $10^{-10}$ près, et les éléments au-dessus de la diagonale de $\mathbf{L}$ sont **exactement** nuls ;
- c) `[is_positive_definite(M) for M in [[[5, 2, 0], [2, 3, 1], [0, 1, 4]], [[2, 4], [4, 7]], [[1, 0], [0, 0]], [[2, 1], [0, 2]], [[6]]]]` (une liste de 5 booléens) ;
- d) le nom de l'erreur levée par `cholesky([[1, 2], [3, 4]])` ;
- e) le nom de l'erreur levée par `cholesky([[1, 3], [3, 4]])` ;
- f) le nom de l'erreur levée par `is_positive_definite([[1, 0, 0], [0, 1, 0]])` ;
- g) vrai ou faux : `type(is_positive_definite(...))` est `bool` pour la première et la deuxième matrice de c).

Dans ta copie : retrouve a) à la main. Pour chaque matrice de c), dis pourquoi elle est, ou n'est pas, définie positive (et laquelle n'est pas symétrique). Pourquoi d) et e) lèvent-elles la même erreur pour deux raisons différentes ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")  # reload your saved file

In [ ]:
A_28r = random_spd(6, 28)
with checked("M12c.28a"):
    wbm.check("M12c.28a", np.asarray(mymath.eigen.cholesky([[25, -10, 15], [-10, 5, -4], [15, -4, 17]])).tolist(), computed=True)
with checked("M12c.28b"):
    L_28 = np.asarray(mymath.eigen.cholesky(A_28r))
    wbm.check("M12c.28b", bool(np.allclose(L_28 @ L_28.T, A_28r, rtol=0, atol=1e-10) and np.all(np.triu(L_28, 1) == 0)),
              computed=True)
with checked("M12c.28c"):
    wbm.check("M12c.28c", [mymath.eigen.is_positive_definite(M) for M in [[[5, 2, 0], [2, 3, 1], [0, 1, 4]], [[2, 4], [4, 7]], [[1, 0], [0, 0]], [[2, 1], [0, 2]], [[6]]]], computed=True)
with checked("M12c.28d"):
    wbm.check("M12c.28d", raised(mymath.eigen.cholesky, [[1, 2], [3, 4]]), computed=True)
with checked("M12c.28e"):
    wbm.check("M12c.28e", raised(mymath.eigen.cholesky, [[1, 3], [3, 4]]), computed=True)
with checked("M12c.28f"):
    wbm.check("M12c.28f", raised(mymath.eigen.is_positive_definite, [[1, 0, 0], [0, 1, 0]]), computed=True)
with checked("M12c.28g"):
    wbm.check("M12c.28g", all(type(mymath.eigen.is_positive_definite(M)) is bool for M in [[[5, 2, 0], [2, 3, 1], [0, 1, 4]], [[2, 4], [4, 7]], [[1, 0], [0, 0]], [[2, 1], [0, 2]], [[6]]][:2]),
              computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12c_eigen.py", "test_cholesky_ or test_is_positive_definite_ or test_eigen_uses_")

### Ex M12c.29 — La racine carrée d'une matrice symétrique positive et la matrice de blanchiment · `sqrtm_psd, whitening_matrix` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la racine carrée d'une matrice symétrique positive par le théorème spectral, et les blanchiments ZCA et de Cholesky.  
**Prérequis :** fiche §6 et §8 ; Ex M12c.28 · **mymath :** `eigen.py` · **Parcours :** C, E

Écris, en suivant les docstrings (ici, et seulement ici, `np.linalg.eigh` est permis) :
- `sqrtm_psd(A, tol=1e-12)` (fiche §8) : `ValueError` si $\mathbf{A}$ n'est pas carrée ou pas symétrique ; `np.linalg.eigh` ; une `ValueError` si une valeur propre est inférieure à $-\mathrm{tol} \cdot \max(1, \lambda_{\max})$ ; les petites valeurs propres négatives (des arrondis) remplacées par 0 ; puis $\mathbf{Q}\operatorname{diag}(\sqrt{\lambda_i})\mathbf{Q}^\top$ ;
- `whitening_matrix(cov, method="zca")` (fiche §8) : `ValueError` si la méthode n'est ni `"zca"` ni `"cholesky"`, si `cov` n'est pas carrée ou pas symétrique ; pour `"zca"`, $\mathbf{Q}\operatorname{diag}(1/\sqrt{\lambda_i})\mathbf{Q}^\top$ (une `ValueError` si une valeur propre est $\le 0$) ; pour `"cholesky"`, $\mathbf{L}$ = `cholesky(cov)` (ta fonction de M12c.28), puis $\mathbf{W}$ solution de $\mathbf{L}\mathbf{W} = \mathbf{I}$ par `np.linalg.solve`.

La vérification calcule :
- a) `sqrtm_psd([[17, 6], [6, 5]])` (une matrice, 6 décimales) ;
- b) `sqrtm_psd([[9, 3], [3, 1]])` (une matrice, 6 décimales) ;
- c) `whitening_matrix([[4, -2], [-2, 2]])` (une matrice, 6 décimales) ;
- d) `whitening_matrix([[4, -2], [-2, 2]], method="cholesky")` (une matrice, 6 décimales) ;
- e) vrai ou faux : pour des données $\mathbf{X}$ corrélées de forme $(500, 3)$ (cellule de vérification), les données blanchies `(X - X.mean(axis=0)) @ W.T` ont pour covariance (`np.cov`) l'identité à $10^{-9}$ près, avec les deux méthodes, quand `W = whitening_matrix(np.cov(X, rowvar=False), method=...)` ;
- f) le nom de l'erreur levée par `whitening_matrix([[4, 0], [0, 1]], method="pca")` ;
- g) le nom de l'erreur levée par `sqrtm_psd([[1, 3], [3, 4]])`.

Dans ta copie : vérifie a) par le carré. Pourquoi b) existe-t-il, alors que $\begin{pmatrix} 9 & 3 \\ 3 & 1 \end{pmatrix}$ n'est pas inversible ? Écris la matrice $\mathbf{L}$ de d) et vérifie $\mathbf{W}\boldsymbol{\Sigma}\mathbf{W}^\top = \mathbf{I}$ à la main. Pourquoi c) est-elle symétrique et pas d) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")  # reload your saved file

In [ ]:
X_29 = np.random.default_rng(29).normal(size=(500, 3)) @ np.array([[2.0, 0.0, 0.0], [1.0, 1.5, 0.0], [-0.5, 1.0, 0.5]])
cov_29 = np.cov(X_29, rowvar=False)
with checked("M12c.29a"):
    wbm.check("M12c.29a", np.asarray(mymath.eigen.sqrtm_psd([[17, 6], [6, 5]])).tolist(), computed=True)
with checked("M12c.29b"):
    wbm.check("M12c.29b", np.asarray(mymath.eigen.sqrtm_psd([[9, 3], [3, 1]])).tolist(), computed=True)
with checked("M12c.29c"):
    wbm.check("M12c.29c", np.asarray(mymath.eigen.whitening_matrix([[4, -2], [-2, 2]])).tolist(), computed=True)
with checked("M12c.29d"):
    wbm.check("M12c.29d", np.asarray(mymath.eigen.whitening_matrix([[4, -2], [-2, 2]], method="cholesky")).tolist(),
              computed=True)
with checked("M12c.29e"):
    whitened_29 = [np.cov((X_29 - X_29.mean(axis=0)) @ np.asarray(mymath.eigen.whitening_matrix(cov_29, method=m)).T,
                          rowvar=False) for m in ("zca", "cholesky")]
    wbm.check("M12c.29e", all(np.allclose(C, np.eye(3), rtol=0, atol=1e-9) for C in whitened_29), computed=True)
with checked("M12c.29f"):
    wbm.check("M12c.29f", raised(mymath.eigen.whitening_matrix, [[4, 0], [0, 1]], method="pca"), computed=True)
with checked("M12c.29g"):
    wbm.check("M12c.29g", raised(mymath.eigen.sqrtm_psd, [[1, 3], [3, 4]]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m12c_eigen.py", "test_sqrtm_psd_ or test_whitening_matrix_ or test_eigen_uses_")

### Ex M12c.30 — `np.linalg.eigh`, `det`, `cholesky` et `scipy.linalg.sqrtm` contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** utiliser `np.linalg.eigh`, `det`, `cholesky` et `scipy.linalg.sqrtm`, lire ce qu'elles rendent, et comparer avec ta librairie.  
**Prérequis :** fiche §1, §6 et §7 ; Ex M12c.26 à M12c.29 · **Parcours :** C, E

En pratique, on ne réécrit pas ces fonctions :
- `np.linalg.eigh(S)` (matrices **symétriques**) rend `(valeurs, vecteurs)` : les valeurs propres réelles par ordre **croissant**, et une matrice dont les **colonnes** sont des vecteurs propres orthonormés ; `np.linalg.eig(A)` (toute matrice carrée) rend la même chose sans ordre garanti, éventuellement en nombres complexes ;
- `np.linalg.det(A)` et `np.linalg.cholesky(S)` (qui ne vérifie pas la symétrie : elle ne lit que le triangle inférieur) ;
- `scipy.linalg.sqrtm(A)`, la racine carrée de toute matrice qui en a une.

Avec $\mathbf{S}$ = `[[6, 2, 1, 0], [2, 5, 2, 1], [1, 2, 4, 2], [0, 1, 2, 3]]`, calcule :
- a) les valeurs propres rendues par `np.linalg.eigh(S)` (une liste, 4 décimales, dans l'ordre rendu) ;
- b) vrai ou faux : les colonnes de la matrice de vecteurs propres rendue par `eigh` sont orthonormées, à $10^{-12}$ près ;
- c) vrai ou faux : la valeur rendue par ta fonction `power_iteration(S)` est la plus grande valeur de a), à $10^{-8}$ près ;
- d) vrai ou faux : ta fonction `det(S)` est égale à `np.linalg.det(S)` et au produit des valeurs de a), à $10^{-10}$ près, relativement ;
- e) vrai ou faux : ta fonction `cholesky(S)` est égale à `np.linalg.cholesky(S)` à $10^{-12}$ près ;
- f) vrai ou faux : ta fonction `sqrtm_psd(S)` est égale à `scipy.linalg.sqrtm(S)` à $10^{-10}$ près ;
- g) le type (`dtype`, en texte : `str(values.dtype)`) des valeurs propres rendues par `np.linalg.eig([[0, -1], [1, 0]])`.

Dans ta copie : pourquoi `eigh` plutôt que `eig` pour une matrice symétrique (deux raisons) ? Que rendrait `np.linalg.cholesky([[4, 7], [-3, 5]])`, et pourquoi est-ce dangereux ? Pourquoi g) n'est-il pas un type de nombres réels ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")
import scipy.linalg

S_30 = np.array([[6, 2, 1, 0], [2, 5, 2, 1], [1, 2, 4, 2], [0, 1, 2, 3]], dtype=float)
answer_M12c_30a = ...  # a) a list of 4 numbers (4 decimals), in the order returned by eigh
answer_M12c_30b = ...  # b) True or False
answer_M12c_30g = ...  # g) a text: str(values.dtype)
answer_M12c_30c = answer_M12c_30d = answer_M12c_30e = answer_M12c_30f = ...
# Each block stands alone: compute inside it what it needs, so that "Run all" goes on when a function is not written.
with wbm.attempt("M12c.30c"):        # your power_iteration is called here: ⏳ while it is not written
    answer_M12c_30c = ...  # c) True or False
with wbm.attempt("M12c.30d"):        # your det
    answer_M12c_30d = ...  # d) True or False
with wbm.attempt("M12c.30e"):        # your cholesky
    answer_M12c_30e = ...  # e) True or False
with wbm.attempt("M12c.30f"):        # your sqrtm_psd
    answer_M12c_30f = ...  # f) True or False

In [ ]:
for letter, answer in zip("abcdefg", [answer_M12c_30a, answer_M12c_30b, answer_M12c_30c, answer_M12c_30d,
                                       answer_M12c_30e, answer_M12c_30f, answer_M12c_30g]):
    wbm.check(f"M12c.30{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §5 à §7. La méthode de la puissance quand les valeurs propres sont proches, sa vitesse, et les lignes de niveau des formes quadratiques.

### Ex M12c.31 — Prédire : la méthode de la puissance quand les deux plus grandes valeurs propres sont proches 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la vitesse de la méthode de la puissance et la précision du vecteur et de la valeur propre quand les deux plus grandes valeurs propres sont proches, puis le vérifier.  
**Prérequis :** fiche §5 et §6 ; Ex M12c.27 · **Parcours :** C, E

La cellule fournie définit deux matrices diagonales : $\operatorname{diag}(10, 5, 1)$, dont les deux plus grandes valeurs propres sont **éloignées** ($|\lambda_2/\lambda_1| = 0{,}5$), et $\operatorname{diag}(10 ;\ 9{,}9 ;\ 1)$, dont elles sont **proches** ($0{,}99$). On lance `power_iteration` sur chacune depuis $(1, 1, 1)$, avec les réglages par défaut (`n_iter=1000`, `tol=1e-12`). Pour chaque question, choisis le numéro de la bonne réponse :
- a) le nombre d'itérations pour la matrice éloignée : (1) moins de 100 ; (2) entre 100 et 999 ; (3) 1000, le maximum, sans atteindre la tolérance ;
- b) le nombre d'itérations pour la matrice proche : mêmes choix ;
- c) pour la matrice proche, l'écart entre la valeur rendue et 10 : (1) moins de $10^{-6}$ ; (2) entre $10^{-6}$ et $10^{-2}$ ; (3) plus de $10^{-2}$ ;
- d) pour la matrice proche, l'écart $\lVert \mathbf{v} - \mathbf{e}_1 \rVert$ du vecteur rendu : (1) moins de $10^{-8}$ ; (2) entre $10^{-8}$ et $10^{-3}$ ; (3) plus de $10^{-3}$.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

In [ ]:
A_FAR_31 = np.diag([10.0, 5.0, 1.0])       # λ₂ / λ₁ = 0.5
A_CLOSE_31 = np.diag([10.0, 9.9, 1.0])     # λ₂ / λ₁ = 0.99
START_31 = np.ones(3)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M12c_31a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M12c_31b = ...  # b) 1, 2 or 3
prediction_M12c_31c = ...  # c) 1, 2 or 3
prediction_M12c_31d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M12c_31a, prediction_M12c_31b, prediction_M12c_31c,
                                      prediction_M12c_31d]):
    wbm.check(f"M12c.31{letter}", prediction)

**L'expérience** (avec ta fonction `power_iteration`, ou la référence si elle n'est pas encore écrite). Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M12c_31a, prediction_M12c_31b, prediction_M12c_31c, prediction_M12c_31d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")
    try:                                             # your power_iteration if it runs, else the reference
        mymath.eigen.power_iteration(A_FAR_31, x0=START_31)
        power_31 = mymath.eigen.power_iteration
        print("avec ta fonction power_iteration")
    except Exception:  # noqa: BLE001 - not written yet (NotImplementedError), missing file, or a bug
        power_31 = wbm.load_mymath("ref", alias="mymath_ref").eigen.power_iteration
        print("ℹ️ ta fonction power_iteration n'est pas encore écrite (ou lève une erreur) : l'expérience utilise la référence")
    for name, A in (("éloignées", A_FAR_31), ("proches", A_CLOSE_31)):
        value, vector, iterations = power_31(A, x0=START_31)
        print(f"{name:10s} : {iterations:5d} itérations, valeur {value:.12f}, écart du vecteur à e₁ "
              f"{np.linalg.norm(vector - np.array([1.0, 0.0, 0.0])):.1e}")

**Après l'expérience**, dans ta copie : combien d'itérations aurait-il fallu à la matrice proche pour atteindre la tolérance (résous $0{,}99^k \approx 10^{-12}$) ? Pourquoi la valeur propre est-elle si précise alors que le vecteur ne l'est pas (écris le quotient de Rayleigh avec les coordonnées de $\mathbf{x}$, fiche §6) ? Qu'en conclure sur un critère d'arrêt qui ne regarde que la valeur propre ?

### Ex M12c.32 — La vitesse de la méthode de la puissance et le rapport |λ₂ / λ₁| 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le nombre d'itérations de la méthode de la puissance en fonction du rapport des deux plus grandes valeurs propres, et retrouver la loi $|\lambda_2/\lambda_1|^k$.  
**Prérequis :** fiche §5 ; Ex M12c.27 et M12c.31 · **Parcours :** C

Pour chaque rapport $r$ de `RATIOS_32`, la fonction fournie `ratio_matrix(r)` rend $\operatorname{diag}(1, r, r/2)$, dont les deux plus grandes valeurs propres ont le rapport $r$. Écris :
- `iteration_table()` : la liste des nombres d'itérations de `mymath.eigen.power_iteration(ratio_matrix(r), x0=np.ones(3), n_iter=10_000)` (tolérance par défaut, $10^{-12}$), pour chaque $r$ de `RATIOS_32`.

La cellule suivante affiche le tableau, avec le produit $k \times (-\log_{10} r)$ : le nombre de chiffres décimaux gagnés en $k$ pas si l'écart est multiplié par $r$ à chaque pas. Lis-le, puis réponds :
- a) le nombre d'itérations pour $r = 0{,}5$ ;
- b) le plus petit $r$ du tableau qui demande plus de 1 000 itérations ;
- c) vrai ou faux : pour tous les $r$ du tableau, $k \times (-\log_{10} r)$ est compris entre 9 et 14 ;
- d) vrai ou faux : passer de $r = 0{,}9$ à $r = 0{,}99$ multiplie le nombre d'itérations par plus de 9.

Dans ta copie : pourquoi le produit $k \times (-\log_{10} r)$ est-il presque constant, et que vaut-il en théorie pour une tolérance de $10^{-12}$ ? Pourquoi $r = 0{,}1$ ne demande-t-il pas exactement 12 itérations ? Pour une matrice $10^6 \times 10^6$ avec $r = 0{,}999$, combien de produits matrice-vecteur faudrait-il, et que faire à la place (fiche §5, 💻) ?

In [ ]:
RATIOS_32 = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]


def ratio_matrix(r):
    """A 3 × 3 diagonal matrix whose two largest eigenvalues have the ratio r: diag(1, r, r / 2)."""
    return np.diag([1.0, r, r / 2])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")


def iteration_table():
    """For r in RATIOS_32: the number of iterations of power_iteration on ratio_matrix(r), from (1, 1, 1)."""
    raise NotImplementedError("une liste de 7 entiers : le troisième élément rendu par power_iteration")

In [ ]:
with checked("M12c.32 tableau"):
    counts = iteration_table()
    print("  r      itérations   itérations × (−log₁₀ r)")
    for r, k in zip(RATIOS_32, counts):
        print(f"{r:5.2f}   {k!s:>10}   {k * -np.log10(r):10.1f}")
    if 1000 in counts:
        print("⚠️ un nombre d'itérations vaut 1000, la valeur par défaut de n_iter : as-tu passé n_iter=10_000 ?")

answer_M12c_32a = ...  # a) an integer, read in the table above
answer_M12c_32b = ...  # b) a number of RATIOS_32
answer_M12c_32c = ...  # c) True or False
answer_M12c_32d = ...  # d) True or False
for letter, answer in zip("abcd", [answer_M12c_32a, answer_M12c_32b, answer_M12c_32c, answer_M12c_32d]):
    wbm.check(f"M12c.32{letter}", answer)

### Ex M12c.33 — Les lignes de niveau de xᵀAx : définie positive, semi-définie positive, indéfinie 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner les lignes de niveau d'une forme quadratique avec ses axes propres, et classer une matrice symétrique d'après les signes de ses valeurs propres.  
**Prérequis :** fiche §6 et §7 · **Parcours :** C, E

Écris :
- `classify(A)` : pour une matrice symétrique $2 \times 2$, le texte `"définie positive"`, `"semi-définie positive"`, `"définie négative"`, `"semi-définie négative"` ou `"indéfinie"`, d'après les signes de ses valeurs propres (`np.linalg.eigvalsh`) ; une valeur propre de valeur absolue inférieure à $10^{-12}$ fois la plus grande compte comme nulle ;
- `draw_levels(ax, A)` : dans l'axe `ax`, les lignes de niveau de $q(\mathbf{x}) = \mathbf{x}^\top\mathbf{A}\mathbf{x}$ sur $[-2 ; 2]^2$ (`ax.contour` sur une grille `np.meshgrid`, avec quelques niveaux positifs et, si la forme en prend, négatifs), et les deux vecteurs propres unitaires de $\mathbf{A}$ en flèches (`ax.annotate` ou `ax.arrow`), avec `ax.set_aspect("equal")`.

La cellule de la figure s'en sert pour les trois matrices fournies ; une cellule de vérification essaie `classify` sur sept autres matrices.

Dans ta copie : pour la matrice définie positive, le long de quel vecteur propre l'ellipse est-elle la plus allongée, et pourquoi (fiche §7) ? Pour la matrice semi-définie, dans quelle direction $q$ reste-t-elle nulle, et quel est le lien avec son noyau ? Pour l'indéfinie, le long de quelles droites $q$ s'annule-t-elle ? Pourquoi $\operatorname{diag}(1 ;\ 10^{-9})$ est-elle « définie positive » pour `classify`, alors qu'à l'écran elle ressemble à la semi-définie ?

In [ ]:
MATRICES_33 = {"définie positive": np.array([[3.0, -1.0], [-1.0, 4.0]]),
               "semi-définie positive": np.array([[1.0, 3.0], [3.0, 9.0]]),
               "indéfinie": np.array([[1.0, 2.0], [2.0, -2.0]])}

In [ ]:
def classify(A):
    """The kind of the symmetric 2 × 2 matrix A, from the signs of its eigenvalues (zero within 1e-12 · max |λ|)."""
    raise NotImplementedError("np.linalg.eigvalsh, puis les signes")


def draw_levels(ax, A):
    """In ax: the level curves of xᵀAx on [-2, 2]², and the two unit eigenvectors of A as arrows."""
    raise NotImplementedError("np.meshgrid, ax.contour, puis deux flèches")

In [ ]:
with checked("M12c.33 figure"):
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6))
    try:
        for ax, (name, A) in zip(axes, MATRICES_33.items()):
            draw_levels(ax, A)
            ax.set_title(f"{name} : {classify(A)}")
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Vérifications automatiques** (sept autres matrices pour `classify`).

In [ ]:
with checked("M12c.33 vérifications"):
    cases_33 = [([[2.0, 0.0], [0.0, 5.0]], "définie positive"), ([[0.0, 0.0], [0.0, 3.0]], "semi-définie positive"),
                ([[-2.0, 1.0], [1.0, -3.0]], "définie négative"), ([[0.0, 1.0], [1.0, 0.0]], "indéfinie"),
                ([[2.0, 6.0], [6.0, 18.0]], "semi-définie positive"), ([[1.0, 0.0], [0.0, 1e-9]], "définie positive"),
                ([[-0.1, -0.3], [-0.3, -0.9]], "semi-définie négative")]
    wrong_33 = [(A, expected, classify(np.array(A))) for A, expected in cases_33 if classify(np.array(A)) != expected]
    if wrong_33:
        A, expected, got = wrong_33[0]
        print(f"❌ Ex M12c.33 : classify({A}) rend {got!r}, on attendait {expected!r} : regarde les signes des valeurs "
              f"propres (np.linalg.eigvalsh), avec une tolérance pour zéro.")
    else:
        print("✅ Ex M12c.33 : classify range correctement sept matrices, dont trois semi-définies.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §3 à §7. Trois erreurs à trouver, une vérification avec SymPy, puis les trois plus grandes valeurs propres d'une grande matrice.

### Ex M12c.34 — Chasse à l'erreur : det(A + B) = det A + det B, un vecteur propre nul, Cholesky d'une matrice non symétrique 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur le déterminant, les vecteurs propres et Cholesky, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1, §3 et §7 · **Parcours :** C

Un collègue a écrit le code suivant ; six lignes portent une lettre et un commentaire qui dit ce qu'elles doivent calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
A = np.array([[1.0, 2.0], [3.0, 1.0]])
B = np.array([[2.0, 0.0], [1.0, -1.0]])
C = np.array([[4.0, 2.0], [-2.0, 5.0]])
x = np.array([1.0, 1.0])
d = np.linalg.det(A) + np.linalg.det(B)                   # (A) det(A + B)
t = np.trace(A) + np.trace(B)                             # (B) tr(A + B)
lam = 1 + np.sqrt(6)                                      # an eigenvalue of A
v = np.linalg.solve(A - lam * np.eye(2), np.zeros(2))     # (C) an eigenvector of A for lam: v[1] / v[0]
p = np.prod(np.linalg.eigvals(A))                         # (D) det A
L = np.linalg.cholesky(C)                                 # (E) the Cholesky factor of C, with C = L L^T: L[1, 0]
r = (x @ A @ x) / (x @ x)                                 # (F) the Rayleigh quotient of A at x
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour les lignes (A), (B), (C), (D) et (F), la bonne valeur de ce que dit leur commentaire (une liste de 5 nombres, 4 décimales) ;
- c) vrai ou faux : `np.linalg.cholesky(C)` lève une erreur.

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Que rend la ligne (C), et pourquoi un résultat de ce genre devrait-il toujours alerter ? Que factorise réellement la ligne (E) ? Laquelle des trois est la plus dangereuse, et pourquoi ?

In [ ]:
answer_M12c_34a = ...  # a) letters, as text
answer_M12c_34b = ...  # b) a list of 5 values (4 decimals)
answer_M12c_34c = ...  # c) True or False

In [ ]:
wbm.check("M12c.34a", answer_M12c_34a)
wbm.check("M12c.34b", answer_M12c_34b)
wbm.check("M12c.34c", answer_M12c_34c)

### Ex M12c.35 — Vérifier avec SymPy : polynôme caractéristique, valeurs propres exactes, diagonalisation 🛠️ ★ ⏱️ 15 min
**Objectif :** vérifier un calcul de valeurs propres et une diagonalisation avec SymPy, en valeurs exactes, et reconnaître une matrice défective.  
**Prérequis :** fiche §3 et §4 ; R0 (SymPy) · **Parcours :** C

SymPy calcule en valeurs **exactes** : racines carrées et fractions restent écrites telles quelles, ce qui permet de vérifier un calcul papier sans arrondi. Avec les deux matrices fournies, écris :
- `charpoly_35` : le polynôme caractéristique $\det(\mathbf{M} - \lambda\mathbf{I})$ de `M35_DIAG`, en fonction du symbole `lam` (`(M35_DIAG - lam * sympy.eye(3)).det()`, puis `sympy.factor` pour le lire) ;
- `eigenvalues_35` : la liste de ses trois valeurs propres exactes (`sympy.solve(charpoly_35, lam)`, ou `M35_DIAG.eigenvals()`) ;
- `P_35, D_35` : la diagonalisation `M35_DIAG.diagonalize()` ;
- `diagonalizable_35` : `M35_DEFECTIVE.is_diagonalizable()`.

La cellule de vérification contrôle tes quatre résultats (✅ ou ❌, sans `wbm.check`).

Dans ta copie : écris les valeurs propres exactes de `M35_DIAG`, et vérifie que leur somme et leur produit sont la trace et le déterminant. Que rend `M35_DEFECTIVE.eigenvects()`, et pourquoi la matrice n'est-elle pas diagonalisable ? Pourquoi ne faut-il pas utiliser SymPy pour une matrice $100 \times 100$ ?

In [ ]:
import sympy

M35_DIAG = sympy.Matrix([[2, 1, 0], [1, 2, 1], [0, 1, 2]])
M35_DEFECTIVE = sympy.Matrix([[2, 0, 1], [1, 3, 1], [1, 0, 2]])
lam = sympy.symbols("lambda")

In [ ]:
charpoly_35 = ...     # the characteristic polynomial of M35_DIAG, in lam
eigenvalues_35 = ...  # a list of 3 exact eigenvalues
P_35 = D_35 = None    # the diagonalization of M35_DIAG
diagonalizable_35 = ...  # M35_DEFECTIVE.is_diagonalizable()

In [ ]:
with checked("M12c.35 vérifications"):
    problems_35, pending_35 = [], []
    reference_35 = (M35_DIAG - lam * sympy.eye(3)).det()
    if charpoly_35 is ...:
        pending_35.append("charpoly_35")
    else:
        expression_35 = charpoly_35.as_expr() if hasattr(charpoly_35, "as_expr") else charpoly_35
        if sympy.expand(expression_35 + reference_35) == 0:
            problems_35.append("charpoly_35 vaut det(λI − M), ce que rend .charpoly() : en dimension 3, c'est l'opposé de "
                               "det(M − λI) ; écris (M35_DIAG - lam * sympy.eye(3)).det()")
        elif sympy.expand(expression_35 - reference_35) != 0:
            problems_35.append("charpoly_35 doit être det(M − λI), en fonction du symbole lam")
    if eigenvalues_35 is ...:
        pending_35.append("eigenvalues_35")
    else:
        try:
            ok_values_35 = (sorted(eigenvalues_35, key=lambda e: float(e))
                            == sorted(sympy.roots(reference_35, lam).keys(), key=lambda e: float(e)))
        except (TypeError, ValueError):
            ok_values_35 = False
        if not ok_values_35:
            problems_35.append("eigenvalues_35 : les trois valeurs propres exactes de M35_DIAG (sympy.solve ou .eigenvals())")
    if P_35 is None or D_35 is None or P_35 is ... or D_35 is ...:
        pending_35.append("P_35, D_35")
    elif sympy.simplify(P_35 * D_35 * P_35.inv() - M35_DIAG) != sympy.zeros(3):
        problems_35.append("P_35 et D_35 : M35_DIAG.diagonalize() rend (P, D) avec M = P D P⁻¹")
    if diagonalizable_35 is ...:
        pending_35.append("diagonalizable_35")
    elif diagonalizable_35 is not False:
        problems_35.append("diagonalizable_35 : M35_DEFECTIVE.is_diagonalizable()")
    if problems_35:
        print("❌ Ex M12c.35 : à revoir : " + " ; ".join(problems_35) + "."
              + (f" (Pas encore fait : {', '.join(pending_35)}.)" if pending_35 else ""))
    elif pending_35:
        print("⏳ Ex M12c.35 : pas encore fait : " + ", ".join(pending_35) + ".")
    else:
        print("✅ Ex M12c.35 : polynôme caractéristique, valeurs propres exactes, diagonalisation et cas défectif vérifiés.")

### Ex M12c.36 — Défi : les trois plus grandes valeurs propres d'une matrice symétrique semi-définie positive 300 × 300, par la puissance avec déflation, à 1e−6 près 🏆 ★★★ ⏱️ 30 min
**Objectif :** trouver les plus grandes valeurs propres d'une grande matrice symétrique avec des produits matrice-vecteur seulement, par la méthode de la puissance et la déflation.  
**Prérequis :** fiche §5 et §6 ; Ex M12c.27 et M12c.32 · **Parcours :** C

La fonction fournie `make_spectrum_matrix(seed)` fabrique une matrice symétrique semi-définie positive $300 \times 300$, dont les trois plus grandes valeurs propres valent 10, 8 et 6,5, et les 297 autres sont entre 0 et 5 (tu ne connais pas ses vecteurs propres).

**L'objectif** : écrire `top_three(A)`, qui rend la liste de ses trois plus grandes valeurs propres, par ordre décroissant, à moins de $10^{-6}$ de celles de `np.linalg.eigvalsh` sur les trois matrices `seed` = 36, 37 et 38, **sans** `np.linalg.eigh`, `eigvalsh`, `eig`, `eigvals`, `svd`, ni `scipy`. Une dernière cellule enregistre si l'objectif est atteint.

Pistes (fiche §5 et §6) : la méthode de la puissance (ta fonction `power_iteration`, ou ta propre boucle) donne $\lambda_1$ et $\mathbf{q}_1$ ; la **déflation** $\mathbf{A} \leftarrow \mathbf{A} - \lambda_1\mathbf{q}_1\mathbf{q}_1^\top$ (produit extérieur : `np.outer`) remplace $\lambda_1$ par 0 sans changer les autres ; recommence. Pense au point de départ (pas orthogonal au vecteur cherché) et au nombre d'itérations : quel est le rapport $|\lambda_2/\lambda_1|$ à chaque étape ?

Dans ta copie : combien d'itérations demande chaque étape ? Laquelle devrait être la plus lente d'après les rapports $|\lambda_2/\lambda_1|$ de chaque étape, et l'observes-tu sur les trois matrices (le point de départ compte aussi) ? Pourquoi la déflation ne marcherait-elle pas telle quelle sur une matrice non symétrique ? Comment trouverais-tu la plus **petite** valeur propre avec la même méthode (indice : $c\mathbf{I} - \mathbf{A}$) ?

In [ ]:
def make_spectrum_matrix(seed, n=300):
    """A symmetric positive semi-definite (n, n) matrix, of eigenvalues 10, 8, 6.5 and n − 3 others in [0, 5]."""
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
    values = np.concatenate([[10.0, 8.0, 6.5], rng.uniform(0.0, 5.0, size=n - 3)])
    A = (Q * values) @ Q.T
    return (A + A.T) / 2

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M12c")


def top_three(A):
    """The three largest eigenvalues of the symmetric positive semi-definite matrix A, in decreasing order."""
    raise NotImplementedError("la puissance, puis la déflation A − λ q qᵀ, trois fois")

In [ ]:
with checked("M12c.36 défi"):
    import inspect
    import re
    problems, ok_36, ran_36 = [], False, False
    try:
        source_36 = inspect.getsource(top_three)
    except (OSError, TypeError):
        source_36 = ""
    banned_36 = sorted(set(re.findall(r"\b(eigh|eigvalsh|eig|eigvals|svd|scipy)\b", source_36)))
    if banned_36:
        ran_36 = True
        problems.append(f"top_three utilise {', '.join(banned_36)}, interdit dans ce défi : la méthode de la puissance "
                        f"et la déflation seulement")
    for seed_t in ((36, 37, 38) if not banned_36 else ()):
        A_t = make_spectrum_matrix(seed_t)
        expected = np.linalg.eigvalsh(A_t)[::-1][:3]
        try:
            got = np.asarray(top_three(A_t.copy()), dtype=float)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            ran_36 = True
            problems.append(f"top_three lève {type(exc).__name__} ({exc})")
            break
        ran_36 = True
        if got.shape != (3,):
            problems.append(f"matrice {seed_t} : top_three rend la forme {got.shape}, pas (3,)")
            break
        gap = float(np.max(np.abs(got - expected)))
        print(f"matrice {seed_t} : {np.round(got, 8).tolist()}, écart à eigvalsh {gap:.1e}")
        if not gap <= 1e-6:
            problems.append(f"matrice {seed_t} : écart {gap:.1e} à eigvalsh, au-dessus de 1e-6")
            break
    if problems:
        print("❌ Ex M12c.36 : " + " ; ".join(problems) + ".")
    else:
        ok_36 = True
        print("✅ Ex M12c.36 : défi réussi : les trois plus grandes valeurs propres à moins de 1e-6, sur les trois matrices.")
with checked("M12c.36"):
    if not ran_36:                               # ⏳ while top_three is not written
        raise NotImplementedError("top_three, ou une fonction qu'elle appelle, n'est pas encore écrite")
    wbm.check("M12c.36", bool(ok_36), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer un déterminant et l'interpréter comme un volume orienté ?
2. Sais-tu calculer les valeurs et vecteurs propres d'une petite matrice, et diagonaliser ?
3. Sais-tu appliquer le théorème spectral et le quotient de Rayleigh à une matrice symétrique ?
4. Sais-tu programmer et régler la méthode de la puissance ?
5. Sais-tu reconnaître une matrice définie positive, calculer sa décomposition de Cholesky, et blanchir une covariance ?

**La suite** : M12d, la SVD et l'analyse en composantes principales, puis le chapitre 12 de workbookIA (préparation des données).